In [0]:
from pyspark.sql.functions import col
import json

try:

    df = spark.table("weather_catalog.silver.forecasts")

    row_count = df.count()

    duplicate_count = (
        df.groupBy("LocationKey","DailyForecasts_Date")
          .count()
          .filter(col("count") > 1)
          .count()
    )

    null_count = (
        df.filter(
            col("LocationKey").isNull() |
            col("DailyForecasts_Date").isNull()
        ).count()
    )

    expected_locations = (
        spark.table("weather_catalog.gold.location_master")
             .select("key")
             .distinct()
             .count()
    )

    actual_locations = (
        df.select("LocationKey")
          .distinct()
          .count()
    )

    validation = (
        row_count > 0
        and duplicate_count == 0
        and null_count == 0
        and expected_locations == actual_locations
    )

    status = {
        "status":"SUCCESS" if validation else "FAIL",
        "row_count":row_count,
        "duplicate_count":duplicate_count,
        "null_count":null_count,
        "expected_locations":expected_locations,
        "actual_locations":actual_locations
    }

except Exception as e:

    status = {
        "status":"FAIL",
        "error":str(e)
    }

# Exit OUTSIDE the try/except
dbutils.notebook.exit(json.dumps(status))